This is <B>notebook 5</B>. The <B>purpose</B> of this notebook is to explore the <B>selected statistical features impact on salary</B> with respect to <B>era</B>.

## 5. Statistical Inference of NBA Players by Collective Bargaining Agreement Era

The Collective Bargaining Agreement (CBA) between the National Basketball Player's Association (NBPA) and the National Basekball Association (NBA) sets out the terms and conditions of employment for all professional basketball players playing in the National Basketball Association, as well as the respective rights and obligations of the NBA Clubs, the NBA, and the NBPA [1]. The CBA defines the salary cap, the procedures for determining how it is set, the minimum and maximum salaries, the rules for trades, the procedures for the NBA draft, and hundreds of other things that need to be defined in order for a league like the NBA to function. The CBA also prevents the NBA from being in violation of federal antitrust laws. Many of the league's practices (such as the salary cap and draft) would violate antitrust laws were they not agreed to via collective bargaining [2]. The duration of each CBA contract is 7 years.

1. https://nbpa.com/cba

2. http://www.cbafaq.com/salarycap.htm#Q1

In this notebook, we will derive some useful insights based on the players statistics that are extracted by the feature called 'Era'.  The Era feature divides the dataset into two groups: previous (for the 2011 - 2017 contract years) and current (for the 2017- 2023 contract years).

NOTE: The images in this notebook have a different background compared to our technical report for ease of viewing.

We will beign by first importing all the Python packages that will be used throughout the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import altair as alt
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from plotly.subplots import make_subplots

A dataframe df_era is created by reading in the CSV file generated at the end of Notebook 3 (Exploring Dataset for Outliers)

In [ ]:
df_era = pd.read_csv("df_final_no_outliers.csv")

We can use the shape function to find the dimensions of the df_era dataframe. Our dataframe has 6305 individual records and 74 different features.

In [ ]:
df_era.shape

(6305, 74)

It is useful to understand the labels of all the features in our dataframe, which can be obtained by the columns attribute.

In [ ]:
df_era.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS',
       'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'season_id',
       'COUNTRY', 'E_OFF_RATING', 'OFF_RATING', 'sp_work_OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'sp_work_DEF_RATING', 'E_NET_RATING',
       'NET_RATING', 'sp_work_NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO',
       'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'E_TOV_PCT', 'EFG_PCT',
       'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE', 'PACE_PER40',
       'sp_work_PACE', 'PIE', 'POSS', 'FGM_PG', 'FGA_PG', 'Names', 'Positions',
       'AAV+Bonus', 'Year_List', 'Salary', 'Outliers', 'Era'],
      dtype='object')

Based on the analysis perfomed in Notebook 4, the features that we will use for this notebook are specified in the columns_to_keep list and is used to create a reduced dataframe called df_era_red. This primarily reduces our features from 74 to 10 as seen in the shape property. Another list, columns_to_plot, is also created to be used primarily for plotting later in the notebook.

In [ ]:
columns_to_keep = ['FGM_PG', 'AGE', 'FG3A', 'DREB_PCT', 'OREB_PCT', 'DEF_RATING', "Salary", 'PLAYER_ID',"Positions","Era"]

columns_to_plot = ['FGM_PG', 'AGE', 'FG3A', 'DREB_PCT', 'OREB_PCT', 'DEF_RATING', ]

df_era_red = df_era[columns_to_keep]

df_era_red.shape

(6305, 10)

We will use the head function in Python to display the first 5 rows, which is the default setting, of the dataframe. It is always a good practise to get an overview of the data and its structure for the dataframe we are working on. To display more number of rows using the head function, we can specify the desired number in paranthesis. For example, df.head(15) will display the first 15 rows of a dataframe named df. 

In [ ]:
df_era_red.head

<bound method NDFrame.head of       FGM_PG   AGE   FG3A  DREB_PCT  OREB_PCT  DEF_RATING     Salary  \
0        1.3  25.0   88.0     0.092     0.021        97.7  13.460775   
1        1.7  27.0    0.0     0.265     0.105       106.5  14.731801   
2        1.7  27.0    0.0     0.265     0.105       106.5  13.904929   
3        5.4  32.0  303.0     0.181     0.042       102.5  15.715736   
4        5.2  26.0    1.0     0.156     0.077       100.3  16.300417   
...      ...   ...    ...       ...       ...         ...        ...   
6300     4.5  25.0  147.0     0.190     0.076       116.3  15.810211   
6301     8.7  28.0  544.0     0.108     0.016       111.7  17.577453   
6302     2.1  22.0   65.0     0.099     0.087       107.4  14.897543   
6303     2.3  21.0   97.0     0.105     0.028       112.2  15.420334   
6304     9.8  22.0   19.0     0.148     0.060       108.4  16.219549   

      PLAYER_ID       Positions       Era  
0        201985     Point Guard  Previous  
1        201189  

The next code block checks the minimum and maximum values for each of the features we are interested in.

In [ ]:
agg_functions = {
    'FGM_PG': ['min', 'max'],
    'AGE': ['min', 'max'],
    'FG3A': ['min', 'max'],
    'DREB_PCT': ['min', 'max'],
    'OREB_PCT': ['min', 'max'],
    'DEF_RATING': ['min', 'max']
}

agg_result = df_era_red.agg(agg_functions)

print(agg_result)

     FGM_PG   AGE    FG3A  DREB_PCT  OREB_PCT  DEF_RATING
min     0.0  19.0     0.0       0.0       0.0         0.0
max    11.4  43.0  1028.0       1.0       1.0       250.0


It can be seen from the results above that, for the independet features we are interested in utilizing in this notebook, there is a wide variation of values ranging from 0 to 1028 for the feature FG3A while the value range varies from 19 to 43 for the feature AGE.  We will utilize the MinMaxScaler() function available in Python to normalize the range of features between 0 & 1. This aids in allowing all the features to contribute equally to the analysis and also in speeding up the calculations of an algorithm. The columns_to_scale defines the list of independent featues on which the scaling is to be performed. We will first create an instance of the MinMaxScaler class, assigned to scaler, from the scikit-learn library. This is followed by the utilization of the fit_transform method, which fits the scaler to the selected columns. This computes and stores the minimum and maximum values of each independent column and then applies the scaling transformation. A new dataframe, called min_max_df, is then generated using the scaled columns.

In [ ]:
columns_to_scale = ['FGM_PG', 'AGE', 'FG3A', 'DREB_PCT', 'OREB_PCT', 'DEF_RATING']
scaler = MinMaxScaler()
min_max_df = scaler.fit_transform(df_era_red[columns_to_scale])
min_max_df = pd.DataFrame(data = min_max_df, columns = columns_to_scale)

print(min_max_df)

        FGM_PG       AGE      FG3A  DREB_PCT  OREB_PCT  DEF_RATING
0     0.114035  0.250000  0.085603     0.092     0.021      0.3908
1     0.149123  0.333333  0.000000     0.265     0.105      0.4260
2     0.149123  0.333333  0.000000     0.265     0.105      0.4260
3     0.473684  0.541667  0.294747     0.181     0.042      0.4100
4     0.456140  0.291667  0.000973     0.156     0.077      0.4012
...        ...       ...       ...       ...       ...         ...
6300  0.394737  0.250000  0.142996     0.190     0.076      0.4652
6301  0.763158  0.375000  0.529183     0.108     0.016      0.4468
6302  0.184211  0.125000  0.063230     0.099     0.087      0.4296
6303  0.201754  0.083333  0.094358     0.105     0.028      0.4488
6304  0.859649  0.125000  0.018482     0.148     0.060      0.4336

[6305 rows x 6 columns]


We decided to keep the log transformed Salary as exported out in Notebook 3. This was ensure that all the plots that display salary in this notebook are in the same scale as those generated in the other analyses sections of this project. In the next step, we will concatenate the features Salary, PLAYER_ID and Era from the df_era_red dataframe with the newly scaled dataframe min_max_df that was created above.

In [ ]:
df_scaled_concat = pd.concat([min_max_df, df_era_red[['Salary','PLAYER_ID', 'Positions', 'Era']]], axis=1)
print(df_scaled_concat)

        FGM_PG       AGE      FG3A  DREB_PCT  OREB_PCT  DEF_RATING     Salary  \
0     0.114035  0.250000  0.085603     0.092     0.021      0.3908  13.460775   
1     0.149123  0.333333  0.000000     0.265     0.105      0.4260  14.731801   
2     0.149123  0.333333  0.000000     0.265     0.105      0.4260  13.904929   
3     0.473684  0.541667  0.294747     0.181     0.042      0.4100  15.715736   
4     0.456140  0.291667  0.000973     0.156     0.077      0.4012  16.300417   
...        ...       ...       ...       ...       ...         ...        ...   
6300  0.394737  0.250000  0.142996     0.190     0.076      0.4652  15.810211   
6301  0.763158  0.375000  0.529183     0.108     0.016      0.4468  17.577453   
6302  0.184211  0.125000  0.063230     0.099     0.087      0.4296  14.897543   
6303  0.201754  0.083333  0.094358     0.105     0.028      0.4488  15.420334   
6304  0.859649  0.125000  0.018482     0.148     0.060      0.4336  16.219549   

      PLAYER_ID       Posit

We will now use the grouby function in Python to group our data by distinct data points. In the code below, we are first grouping the dataframe by Era followed by the PLAYER_ID and finally by the Positions and we are calculating the mean of the numerical features. 

In [ ]:
df_grouped = df_scaled_concat.groupby(['Era','PLAYER_ID','Positions']).mean().reset_index()

df_grouped.head(10)

,Era,PLAYER_ID,Positions,FGM_PG,AGE,FG3A,DREB_PCT,OREB_PCT,DEF_RATING,Salary
0,Current,1713,Shooting Guard,0.187135,0.958333,0.221466,0.119333,0.0190,0.433067,15.113585
1,Current,1717,Power Forward,0.311404,0.895833,0.263619,0.203500,0.0090,0.434400,15.424948
2,Current,1891,Shooting Guard,0.105263,0.875000,0.111868,0.056000,0.0060,0.422000,14.660800
3,Current,1938,Shooting Guard,0.271930,0.875000,0.186770,0.093000,0.0150,0.422400,14.731801
4,Current,2037,Shooting Guard,0.278947,0.816667,0.240078,0.056600,0.0088,0.460080,15.641636
5,Current,2199,Center,0.116228,0.708333,0.000486,0.209750,0.0985,0.424000,15.520107
6,Current,2200,Center,0.223684,0.770833,0.064689,0.280000,0.0635,0.425400,15.732660
7,Current,2207,Shooting Guard,0.236842,0.750000,0.141051,0.130000,0.0110,0.419600,14.859175
8,Current,2210,Small Forward,0.052632,0.791667,0.006809,0.099000,0.0110,0.441200,14.698063
9,Current,2216,Power Forward,0.535088,0.708333,0.142996,0.215000,0.0630,0.458800,16.300417


In the code below, we will use the scatterplot option in Altair plotting library to visualize the relationship between the different independent features and the dependent feature 'Salary'. It can be noticed that each of the indepent features has a relatively linear relationship with Salary for both Era that are being analyzed.  

In [ ]:
# Code to plot a scatterplot

columns_to_plot = ['FGM_PG', 'AGE', 'FG3A', 'DREB_PCT', 'OREB_PCT', 'DEF_RATING']

scatter_plots = []

for col in columns_to_plot:
    scatter = alt.Chart(df_grouped).mark_circle().encode(
        x=col,
        y='Salary:Q',
        color='Era:N'
    ).properties(
        title=f'Salary vs. {col}',
        width = 300,
        height = 200
    )
    
    scatter_plots.append(scatter)

combined_chart = alt.vconcat(
    alt.hconcat(*scatter_plots[:2]),
    alt.hconcat(*scatter_plots[2:4]),
    alt.hconcat(*scatter_plots[4:]),
    spacing=20
).configure_axis(
        grid=False
)

combined_chart

alt.VConcatChart(...)

Next, we plot the grouped bar chart for each Era by different player position against the Salary. Generally, when a new CBA is signed, the players get a bump in their overall salary. This can be clearly seen below that for every player position, the average current era salary is higher than that of the previous era. Additionally, it appears that the Point Guard position had the largest average salary jump while at the same time the Small Forward position had the lowest salary raise comparitavely.

In [ ]:
# Code to plot a bar chart 

chart = alt.Chart(df_grouped).mark_bar().encode(
    x='Salary:Q',
    y=alt.Y('Era:N', title='Era'),
    color=alt.Color('Era:N'),
    row = 'Positions'
).properties(
    width=500,  
    height=100  
)

bar_chart = chart.configure_axis(
    labelFontSize=12,
    titleFontSize=14
)
bar_chart

alt.Chart(...)

An additional plot that is useful for Exploratory Data Analysis (EDA) is the box & whisker plot. The shape of this box plot shows the distribution of the data and any outliers therein. The center line of the box shows the median value for that box and we can notice that every player position in the Current Era has a higher median compared to the Previous Era. Additionally, for the Small Forward position, we see that in the Current Era, since the median line is closer to the bottom of the box the data is positively skewed (or skewed right) and the cirlce at the top indicates that there is an outlier. The length of the whiskers also indicate the skewness of the data and it can be observed that except for the Point Guard position in the Current Era, every whisker at the top of the box plot have a longer length. This indicates that the data is positively skewed as well.

In [ ]:
# Code to plot a Box * Whisker plot

charts = alt.Chart(df_grouped).mark_boxplot().encode(
    x=alt.X('Era:N', title=None),
    y=alt.Y('Salary:Q', title='Salary', scale=alt.Scale(domain=[12, 18])),
    color=alt.Color('Era:N', legend=None),
    column=alt.Column('Positions:N')
).properties(
    width=100,  
    height=200 
)

box_charts = charts.configure_axis(
    labelFontSize=12,
    titleFontSize=14,
    grid=False
)

box_charts


alt.Chart(...)

Finally, we will be using a Radar plot (also known as irregular polygon, spider chart or web chart) to analyze the effect of the various independent features. This is a very useful plot that can be used in multivariate analysis (generally 3 or more features). The circular chart is divided into equal number of regions based on the total number of features. For example, an analysis with 4 features will divide the the graph in 4 equal regions each containing 90 degrees around the wheel while a 6 features analysis will split the chart into six sectors each of 60 degrees around the circle. Additionally, each radii represents one of each variable and the length of the data in the radii is commesurate to the magnitude of that variable. 

For this analysis, we will look at three of the top 10 Power Forward players in our dataset who have data in both Previous and Current Era. Interrogation of the dataframe resulted in the following three names to be used for this analysis: Giannis Antetokounmpo, Julius Randle and Kristaps Porzingis. 

In [ ]:
# Extracting data for Giannis Antetokounmpo

# Code below obtains the player id for Giannis Antetokounmpo from the df_era dataframe
giannis_df = df_era[df_era['PLAYER_NAME'].str.contains('Giannis Antetokounmpo')]
giannis_player_id = giannis_df['PLAYER_ID'].unique()
print("PLAYER_ID: {}".format(giannis_player_id))

# Code below obtains the features from df_grouped dataframe for Giannis Antetokounmpo
giannis_df_grouped = df_grouped[df_grouped['PLAYER_ID']==203507]
print(giannis_df_grouped)

PLAYER_ID: [203507]
           Era  PLAYER_ID      Positions    FGM_PG       AGE      FG3A  \
252    Current     203507  Power Forward  0.901316  0.229167  0.208414   
1557  Previous     203507  Power Forward  0.471491  0.062500  0.109679   

      DREB_PCT  OREB_PCT  DEF_RATING     Salary  
252     0.2655   0.06050      0.4130  17.034386  
1557    0.1740   0.04425      0.4248  14.582737  


In [ ]:
# Extracting data for Julius Randle

# Code below obtains the player id for Julius Randle from the df_era dataframe
randle_df = df_era[df_era['PLAYER_NAME'].str.contains('Julius Randle')]
randle_player_id = randle_df['PLAYER_ID'].unique()
print("PLAYER_ID: {}".format(randle_player_id))

# Code below obtains the features from df_grouped dataframe for Julius Randle
randle_df_grouped = df_grouped[df_grouped['PLAYER_ID']==203944]
print(randle_df_grouped)

PLAYER_ID: [203944]
           Era  PLAYER_ID      Positions    FGM_PG       AGE      FG3A  \
295    Current     203944  Power Forward  0.656642  0.267857  0.294191   
1628  Previous     203944  Power Forward  0.307018  0.083333  0.032101   

      DREB_PCT  OREB_PCT  DEF_RATING     Salary  
295   0.217714  0.058714    0.443886  16.393976  
1628  0.174333  0.045000    0.471867  15.035305  


In [ ]:
# Extracting data for Kristaps Porzingis

# Code below obtains the player id for Kristaps Porzingis from the df_era dataframe
porzingis_df = df_era[df_era['PLAYER_NAME'].str.contains('Kristaps Porzingis')]
porzingis_player_id = porzingis_df['PLAYER_ID'].unique()
print("PLAYER_ID: {}".format(porzingis_player_id))

# Code below obtains the features from df_grouped dataframe for Kristaps Porzingis
porzingis_df_grouped = df_grouped[df_grouped['PLAYER_ID']==204001]
print(porzingis_df_grouped)

PLAYER_ID: [204001]
           Era  PLAYER_ID      Positions   FGM_PG       AGE      FG3A  \
311    Current     204001  Power Forward  0.65614  0.241667  0.291245   
1653  Previous     204001  Power Forward  0.52193  0.062500  0.270914   

      DREB_PCT  OREB_PCT  DEF_RATING     Salary  
311     0.1984    0.0556     0.44448  16.887228  
1653    0.1715    0.0565     0.42600  15.355067  


The following code plots the Radar chart for each of the three Top 10 Power Forward players. It can be seen for Giannis Antetokounmpo that he made a significant improvement in his FGM_PG statistics between the two eras and this likely led to a noticeably increase in his salary. It can also be observed that the other two players in this particular analysis showed relatively improved FGM_PG statistics as well but not to the extent noticed in Antetokounmpo. This might be an area that they and other players might want to focus on. 

Another area that Antetokounmpo improved was both in his DREB_PCT & FG3A between the two eras while Porzingis made only a slight improvement in these two features, if at all any. On the other hand, while the DREB_PCT was not significant for Randle, he made a tremendous improvement in the FG3A feature. 

The radar plots also show that the salary of the players increase as the players age, which is definitely expected as they become more mature players and their cumulative experiences on the floor enhances their contribution to their teams.

In [ ]:
# Plotting a Radar Chart for three Top 10 Power Forward NBA Players

# fig1 chart is for Giannis Antetokounmpo
fig1 = go.Figure()

fig1.add_trace(go.Scatterpolar(
      r= giannis_df_grouped.iloc[0, 3:9],
      theta=columns_to_plot,
    #   line_close=True,
      fill='toself',
      fillcolor='dark blue',
      line_color='blue',
      name='Current'
))
fig1.add_trace(go.Scatterpolar(
      r= giannis_df_grouped.iloc[1, 3:9],
      theta=columns_to_plot,
    #   line_close=True,
      fill='toself',
      fillcolor='orange',
      line_color='orange',
      name='Previous'
))

fig1.update_layout(
  polar=dict(
    radialaxis=dict(
      visible=True,
      range=[0, 1.0]
    )),
  showlegend=True
)

fig1.update_traces(opacity=.25)

fig1.update_layout(
            title={
            'text' : 'Giannis Antetokounmpo',
            'x':0.5,
            'xanchor': 'center'
        })
fig1.show()

# fig2 chart is for Julius Randle
fig2 = go.Figure()

fig2.add_trace(go.Scatterpolar(
      r= randle_df_grouped.iloc[0, 3:9],
      theta=columns_to_plot,
    #   line_close=True,
      fill='toself',
      fillcolor='dark blue',
      line_color='blue',
      name='Current'
))
fig2.add_trace(go.Scatterpolar(
      r= randle_df_grouped.iloc[1, 3:9],
      theta=columns_to_plot,
    #   line_close=True,
      fill='toself',
      fillcolor='orange',
      line_color='orange',
      name='Previous'
))

fig2.update_layout(
  polar=dict(
    radialaxis=dict(
      visible=True,
      range=[0, 1.0]
    )),
  showlegend=True
)

fig2.update_traces(opacity=.25)

fig2.update_layout(
            title={
            'text' : 'Julius Randle',
            'x':0.5,
            'xanchor': 'center'
        })
fig2.show()

# fig3 chart is for Kristaps Porzingis
fig3 = go.Figure()

fig3.add_trace(go.Scatterpolar(
      r= porzingis_df_grouped.iloc[0, 3:9],
      theta=columns_to_plot,
    #   line_close=True,
      fill='toself',
      fillcolor='dark blue',
      line_color='blue',
      name='Current'
))
fig3.add_trace(go.Scatterpolar(
      r= porzingis_df_grouped.iloc[1, 3:9],
      theta=columns_to_plot,
    #   line_close=True,
      fill='toself',
      fillcolor='orange',
      line_color='orange',
      name='Previous'
))

fig3.update_layout(
  polar=dict(
    radialaxis=dict(
      visible=True,
      range=[0, 1.0]
    )),
  showlegend=True
)

fig3.update_traces(opacity=.25)

fig3.update_layout(
            title={
            'text' : 'Kristaps Porzingis',
            'x':0.5,
            'xanchor': 'center'
        })
fig3.show()

In conclusion, in this notebook we focused on the players aspect as per the CBA era. We divided the dataset into Previous and Current eras and utilized different EDA techniques to explore the data. The three different charts used to understand the population were Scatterplots, Bar charts and Box & Whisker plot. Finally, radar plots were utilized to gain some insights in the player statistics affecting their salary. In future, this piece of the project could be expanded to explore more eras and also look at each of the other 4 player positions.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=0b855501-6f2c-4764-816d-be07593df653' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>